# 7.3 - Cost Engineering: Every Token Has a Price

**Module 7 - LLM API Engineering** | Netsetos GenAI Engineering

This notebook turns a runaway ₹5,10,000/month invoice back into the ₹30,000 estimate it was supposed to be, without dropping quality. You build the four core levers (measure, route, cache, compress), wire in the production layer (observability, stacked provider discounts, hard budgets), and finish with the India-specific line items - GST, forex, and the Indic token tax - that decide the price your finance team actually sees.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup - install the toolkit

One pip line for everything the notebook touches: the token counter, the vector math for the cache, the Gemini SDK, and dotenv for loading keys. It is commented out because it only needs to run once on a fresh Colab or virtualenv.

> **Needs API keys later** - the routing, caching, observability and provider-discount cells make real calls to Gemini, OpenAI, Helicone and Anthropic.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install tiktoken numpy google-genai python-dotenv -q  # noqa


**Explanation**

Environment prep, not code you learn from - it just guarantees the four imports later cells rely on are present.

**How the code works - step by step**
- **`tiktoken`** - OpenAI's tokenizer, used here as an estimate meter for counting and comparing token counts.
- **`numpy`** - the cosine-similarity math behind the semantic cache.
- **`google-genai`** - the current Gemini SDK (the older `google-generativeai` package reached EOL).
- **`python-dotenv`** - loads API keys from a `.env` file so nothing is hardcoded.

**In one line:** one uncomment, four dependencies, run once.

**What you'll see:** (no output - environment prep)

## Setup - load the API key

Keys come from the environment, never from source. `setdefault` seeds an empty `GOOGLE_API_KEY` slot so the later Gemini cells have somewhere to read from - you fill it from your shell or a `.env` file. Any single provider key is enough to start; the multi-provider demos are richer with two or more.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A safety habit, not logic: it establishes the env-var slot the SDKs read from and reminds you never to paste a key inline.

**How the code works - step by step**
- **`import os`** - access to the process environment.
- **`os.environ.setdefault("GOOGLE_API_KEY", "")`** - creates the key only if it is not already set, so an existing real key is never clobbered by the empty default.

**In one line:** reserve the key slot, keep the secret out of the code.

**What you'll see:** (no output - environment prep)

## 1 - The 17x invoice, reconstructed

The whole lesson starts from one real surprise: a ₹30,000 estimate that landed as a ₹5,10,000 bill. This cell is the post-mortem written as comments - it names the five innocent-looking defaults that multiplied into a 17x overrun. No bug was written; every factor was a default nobody questioned.

In [ ]:
# Output: the budget meeting, reconstructed
#
# Your estimate (before launch):
#   500 in + 300 out tokens x 200K conversations x premium model  ~= ₹30,000 / month
#
# The invoice:                                                    ₹5,10,000 / month
#
# Where the 17x came from (the post-mortem):
#   x2.1  multi-turn context - turn 5 re-sends turns 1-4; input grows ~n(n+1)/2
#   x1.4  the "quality check" LLM-judge you added - a full model call, never metered
#   x1.15 retries on timeouts - each one re-sends the WHOLE conversation
#   x1.6  every request on the premium model - including "what are your hours?"
#   x1.3  a 2,000-token system prompt, re-billed on every single request
#
# Nobody wrote a bug. Every factor was a default you never questioned.

**Explanation**

A narrative comment block, not runnable code - it's the invoice that motivates every technique that follows, laid out as a set of multiplying cost factors.

**How the code works - step by step**
- **the estimate line** - 500 in + 300 out tokens x 200K conversations x a premium model, budgeted at ~₹30,000.
- **the multipliers** - multi-turn context re-sending prior turns (~x2.1), an un-metered LLM judge (x1.4), retries that re-send the whole conversation (x1.15), routing everything to premium (x1.6), and a 2,000-token system prompt re-billed every request (x1.3).
- **the punchline comment** - these MULTIPLY, which is how five modest factors reach 17x.

**In one line:** cost engineering is questioning each of those defaults, one per remaining cell.

**What you'll see:** No output - the cell is entirely comments reconstructing the budget-meeting math (estimate vs invoice and the five multipliers). Nothing executes.

## 2 - Count every token

You can't optimize what you don't measure, and the cold-open invoice was a surprise precisely because nobody was reading the meter. This cell installs the meter: a `CostTracker` that counts input and output tokens, prices each call from a date-stamped tariff card, and projects the running total to a daily and monthly bill.

In [ ]:
# TOKEN COUNTING & COST TRACKING
import tiktoken  # OpenAI's tokenizer: fine for OpenAI + format comparisons.
# For Claude/Gemini BILLS, use API-reported response.usage (or their count_tokens
# endpoints) - tiktoken misreads their tokenizers by 15-20%+. Estimates only here.
class CostTracker:
    PRICING = {  # $/M tokens (input, output) - captured 2026-07-02; prices move monthly
        "gemini-2.5-flash-lite":(0.10,0.40), "gpt-5.4-mini":(0.75,4.50),
        "gpt-5.4":(2.50,15.0), "claude-sonnet-4-6":(3.0,15.0)}
    def __init__(self):
        self.enc=tiktoken.get_encoding("cl100k_base"); self.total_cost=0; self.calls=[]
    def count(self,t): return len(self.enc.encode(t))
    def log(self,model,prompt,resp):
        ti,to=self.count(prompt),self.count(resp)
        if model not in self.PRICING: raise KeyError(f"no price for {model} - add it")  # fail loud, never $0-or-guess
        p=self.PRICING[model]; c=(ti*p[0]+to*p[1])/1e6
        self.total_cost+=c; self.calls.append({"model":model,"cost":c}); return c
    def report(self):
        print(f"  {len(self.calls)} calls | Total: ${self.total_cost:.6f}")
        print(f"  Daily (10K): ${self.total_cost/max(len(self.calls),1)*10000:.2f}")
        print(f"  Monthly:     ${self.total_cost/max(len(self.calls),1)*300000:.2f}")

t=CostTracker()
t.log("gemini-2.5-flash-lite","What is Python?","Python is a language."*5)
t.log("gpt-5.4","Explain microservices..."*10,"Microservices..."*50)
t.log("claude-sonnet-4-6","Write a plan..."*20,"Executive summary..."*100)
t.report()

# Output: (representative run)
#   3 calls | Total: $0.004273
#   Daily (10K): $14.24
#   Monthly:     $427.29
# The premium call cost ~30x the flash-lite call for the same round trip -
# and remember: this is the TOKEN bill only (~30-40% of true spend).

**Explanation**

A measurement harness, not a model call - it encodes text locally with tiktoken and multiplies by a per-model price table, so it costs nothing to run. The core idea is that input and output are priced separately (output is the expensive lane) and an unknown model must fail loud rather than silently price at zero.

**How the code works - step by step**
- **`PRICING`** - a date-stamped tariff card of `(input, output)` $/M rates per model; prices move monthly, so the capture date is in the comment.
- **`__init__`** - loads the `cl100k_base` encoder and zeroes the running total and call log.
- **`count()`** - the estimate meter: tiktoken-encodes a string and returns its token length (exact for OpenAI, approximate for Claude/Gemini).
- **`log()`** - one receipt per call: counts both directions, raises `KeyError` on an unpriced model instead of guessing, prices the two lanes separately, and accumulates the total.
- **`report()`** - projects the per-call average to 10K/day and a month, and prints the honest caveat that tokens are only ~30-40% of true spend.

**In one line:** meter first, tariff card second, projection last - and never price what you haven't metered.

**What you'll see:** Three logged calls print a total (~$0.004273), a projected daily figure (~$14.24) and monthly (~$427.29); the premium call costs roughly 30x the flash-lite call for the same round trip, and this is the token bill only.

## 3 - Cheap-first-escalate (the 90/10 strategy, honest version)

The meter shows a 30x price gap between the cheapest and dearest model for the same round trip. The first structural saving: send every query to the cheap model first, and escalate to premium only when a quality gate says the cheap answer isn't good enough. The catch this cell makes honest - the judge is itself a paid model call, so it goes on the ledger.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`) - the router and its judge both call Gemini.

In [ ]:
# CHEAP-FIRST-ESCALATE - honest version: it really escalates, and the judge is metered
from google import genai  # google-generativeai reached EOL 2025-11-30
client = genai.Client()      # reads GOOGLE_API_KEY

CHEAP, PREMIUM, JUDGE = "gemini-2.5-flash-lite", "gemini-2.5-pro", "gemini-2.5-flash-lite"

class CheapFirstRouter:
    def __init__(self):
        self.stats={"cheap":0,"escalated":0,"judge_calls":0}  # the judge is a COST, count it
    def _ask(self, model, prompt):
        return client.models.generate_content(model=model, contents=prompt).text.strip()
    def _good_enough(self, q, r):
        self.stats["judge_calls"] += 1
        verdict = self._ask(JUDGE, f"Rate 1-10: does this answer the query?\nQ:{q[:200]}\nA:{r[:300]}\nReturn ONLY the number.")
        try: return int(verdict.split()[0]) >= 7
        except ValueError: return True   # unparseable judge = don't burn premium money on noise
    def generate(self, query):
        r = self._ask(CHEAP, query)
        if self._good_enough(query, r):
            self.stats["cheap"] += 1; return r, CHEAP
        self.stats["escalated"] += 1
        return self._ask(PREMIUM, query), PREMIUM   # a REAL escalation - the previous version re-asked the cheap model

router = CheapFirstRouter()
for q in ["What is 2+2?", "Translate hello to Hindi", "Design a distributed system for 10M users"]:
    ans, model = router.generate(q)
    print(f"  [{model:22s}] '{q[:42]}' -> {ans[:48]}...")
print(f"\n  {router.stats['cheap']}/{router.stats['cheap']+router.stats['escalated']} served cheap, "
      f"{router.stats['judge_calls']} judge calls (meter them!)")

# Output: (representative run)
#   [gemini-2.5-flash-lite ] 'What is 2+2?' -> 4...
#   [gemini-2.5-flash-lite ] 'Translate hello to Hindi' -> नमस्ते (namaste)...
#   [gemini-2.5-pro        ] 'Design a distributed system for 10M user' -> A system at this scale needs...
#
#   2/3 served cheap, 3 judge calls (meter them!)
# Honest math: every request costs cheap + judge; escalations add premium on top.
# The judge only pays for itself when (escapes prevented x premium price) > judge spend.

**Explanation**

A production routing pattern with an LLM-as-judge gate, and a deliberately honest one: it really escalates to a different (premium) model, and it counts the judge's own calls as a cost. The point is that savings are real only after you subtract what the receptionist charges on every visit.

**How the code works - step by step**
- **`CHEAP / PREMIUM / JUDGE`** - the triage roster (GP, specialist, receptionist); the judge draws a salary, so `stats["judge_calls"]` is tracked.
- **`_ask()`** - the single generate-content call used by every tier.
- **`_good_enough()`** - an LLM judge with a numeric contract ("Return ONLY the number"); a defensive `except` accepts the cheap answer when the judge is unparseable rather than burning premium money.
- **`generate()`** - asks cheap, serves it if the gate passes, otherwise escalates to the REAL premium model (a previous version re-asked the same cheap model - a placebo escalation).
- **threshold `>= 7`** - the quality dial; raise it and more traffic escalates, tune it on your own eval set.

**In one line:** GP first, specialist on referral - and the receptionist's salary is on the books.

**What you'll see:** Two simple queries are served by flash-lite and the hard "design a distributed system" query escalates to gemini-2.5-pro; the summary prints "2/3 served cheap, 3 judge calls (meter them!)" - one judge call per request regardless of outcome.

## 4 - Semantic caching (never pay twice)

Routing cut the price of answers you still had to generate; the bigger win is not generating at all. Most products hear the same question in a hundred phrasings, so an answer you paid for once should be free the second time. This cell matches questions by MEANING (embedding + cosine similarity), not exact text.

> **Needs a Gemini API key** (set `GOOGLE_API_KEY`) - it calls both the embedding model and the generation model.

In [ ]:
# SEMANTIC CACHING - Match by meaning, not exact text
from google import genai  # the current SDK (google-generativeai reached EOL 2025-11-30)
import numpy as np
client = genai.Client()

class SemanticCache:
    def __init__(self,threshold=0.90):  # tune on real traffic: higher = safer, fewer hits
        self.threshold=threshold; self.cache=[]; self.stats={"hits":0,"misses":0}
    def _embed(self,t):
        r = client.models.embed_content(model="gemini-embedding-001", contents=t)
        return np.array(r.embeddings[0].values)
    def query(self,prompt):
        qe=self._embed(prompt)
        for e in self.cache:
            sim=np.dot(qe,e["emb"])/(np.linalg.norm(qe)*np.linalg.norm(e["emb"]))
            if sim>=self.threshold:
                self.stats["hits"]+=1; return e["resp"],True
        resp=client.models.generate_content(model="gemini-2.5-flash", contents=prompt).text.strip()
        self.cache.append({"emb":qe,"resp":resp})
        self.stats["misses"]+=1; return resp,False

c=SemanticCache(threshold=0.90)
for q in ["What is machine learning?","What is ML?","Explain machine learning",
         "What is deep learning?","Tell me about deep learning","How does Python work?"]:
    ans,hit=c.query(q)
    print(f"  {'⚡ HIT (free!)' if hit else '🔄 MISS (LLM)'} '{q}'")
h,m=c.stats["hits"],c.stats["misses"]
print(f"\n  Hit rate: {h}/{h+m} ({h/(h+m)*100:.0f}%) -> {h} free calls")

# Output: (representative run)
#   🔄 MISS (LLM) 'What is machine learning?'
#   ⚡ HIT (free!) 'What is ML?'
#   ⚡ HIT (free!) 'Explain machine learning'
#   🔄 MISS (LLM) 'What is deep learning?'
#   ⚡ HIT (free!) 'Tell me about deep learning'
#   🔄 MISS (LLM) 'How does Python work?'
#
#   Hit rate: 3/6 (50%) -> 3 free calls
# Tradeoff: every point of threshold you trade for hit rate is a point of
# wrong-answer risk - 'What is MLOps?' can false-match 'What is ML?'. The
# animation below has that exact failure. Review borderline hits by hand.

**Explanation**

A meaning-based cache built on vector similarity: it embeds each question, scans stored vectors for a close-enough neighbour, and serves the cached answer on a hit or generates and files a new one on a miss. The tradeoff it makes visible is that every point of threshold you trade for hit rate is a point of wrong-answer risk.

**How the code works - step by step**
- **`__init__(threshold=0.90)`** - sets the similarity cutoff and initializes an empty cache and hit/miss counters.
- **`_embed()`** - turns a question into a vector via `gemini-embedding-001` (a few paise per call - the cache's only running cost).
- **the cosine-similarity loop** - compares the new question against every cached vector; a linear scan is fine for a demo, a vector index replaces it in production.
- **hit path** - similarity at or above threshold returns the stored answer for free.
- **miss path** - pays for one real `gemini-2.5-flash` generation, then files the answer so the next paraphrase is free.

**In one line:** vectorize the question, look for a close-enough neighbour, serve or cook accordingly.

**What you'll see:** Six questions print HIT/MISS: "What is ML?", "Explain machine learning" and "Tell me about deep learning" hit their paraphrases, the rest miss - a 3/6 (50%) hit rate meaning 3 free calls, with a printed warning that a high threshold can false-match (e.g. MLOps -> ML).

## 5 - Prompt compression (fewer tokens, same info)

Caching removes repeat questions; compression shrinks what is left. Every request re-bills your instructions, so a bloated system prompt is a subscription you never meant to buy. This cell takes a chatty support prompt and rewrites it telegraphically, then measures the token drop and translates it into rupees-per-day at scale.

In [ ]:
# PROMPT COMPRESSION - Same info, fewer tokens
import tiktoken, re
enc=tiktoken.get_encoding("cl100k_base")
def ct(t): return len(enc.encode(t))

bloated="""You are a very helpful and friendly customer support assistant for our 
company. Our company is called Netsetos and we are an educational technology company 
based in the beautiful city of Hyderabad, India. We specialize in providing high-quality 
courses in the field of Artificial Intelligence and Machine Learning. When a customer 
asks you a question, please provide a helpful, detailed, and accurate response. Please 
note that our refund policy allows full refunds within 7 days of purchase. After 7 days, 
we offer a 50% refund up to 30 days. After 30 days, no refunds are available.
Customer question: What is your refund policy?"""

compressed="""Role: Netsetos support (edtech, Hyderabad). Scope: AI/ML courses.
Refund: 7d=100%, 8-30d=50%, 30d+=none.
Q: What is your refund policy?"""

print(f"  Original:   {ct(bloated)} tokens")
print(f"  Compressed: {ct(compressed)} tokens")
print(f"  Reduction:  {(1-ct(compressed)/ct(bloated))*100:.0f}%")
print(f"  Same answer. {(1-ct(compressed)/ct(bloated))*100:.0f}% cheaper per call.")
print(f"  At 100K calls/day on GPT-5.4 ($2.50/M in): saves ~${(ct(bloated)-ct(compressed))*100000*2.5/1e6:.0f}/day")

# Output: (real run - tiktoken is exactly right for THIS job: comparing formats)
#   Original:   161 tokens
#   Compressed: 51 tokens
#   Reduction:  68%
#   Same answer. 68% cheaper per call.
#   At 100K calls/day on GPT-5.4 ($2.50/M in): saves ~$27/day

**Explanation**

A pure measurement cell - no model call, just tiktoken counting two versions of the same prompt. This is exactly the job tiktoken is right for: comparing token counts of two text formats, where relative size is what matters.

**How the code works - step by step**
- **`ct()`** - a one-line helper that tiktoken-encodes a string and returns its token count.
- **`bloated`** - a polite, verbose system prompt plus the customer question, full of filler the model doesn't need.
- **`compressed`** - the same role, scope and refund policy compressed to telegraphic lines.
- **the prints** - report original vs compressed token counts, the percent reduction, and the projected daily saving at 100K calls/day on GPT-5.4 input pricing.

**In one line:** meaning stays, filler pays.

**What you'll see:** Prints 161 -> 51 tokens, a 68% reduction ("68% cheaper per call"), and an estimated saving of ~$27/day at 100K calls/day - tiktoken's count is exactly right here because this is a format comparison.

## 6 - Cost observability with Helicone

The four core techniques are installed; operating them at scale starts with visibility. A bill you can only see as one number is a bill you cannot argue with, assign, or alarm on. This cell wraps an OpenAI call with Helicone using a one-line base-URL change so every request is logged with cost, tokens and latency.

> **Needs an OpenAI key and a Helicone key** (set `HELICONE_API_KEY`, plus your OpenAI credentials) and network access.

In [ ]:
# Step 5: wrap an OpenAI call with Helicone for per-call cost/observability.
# One-line change - point base_url at Helicone and pass your key as a header.
import os
from openai import OpenAI

client = OpenAI(
    base_url="https://oai.helicone.ai/v1",  # proxy logs cost, tokens, latency
    default_headers={"Helicone-Auth": f"Bearer {os.environ['HELICONE_API_KEY']}"},
)
try:
    r = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[{"role": "user", "content": "Summarize cost engineering in one line."}],
        max_tokens=40,
        timeout=20,
    )
    u = r.usage
    print(f"in={u.prompt_tokens} out={u.completion_tokens} (logged to Helicone)")
except Exception as e:
    print(f"call failed: {e}")
# Output:
#   in=14 out=22 (logged to Helicone)

**Explanation**

A proxy-integration demo: point the OpenAI client's `base_url` at Helicone and pass an auth header, and the gateway itemizes every call. It's the same OpenAI request you'd make anyway - the only change is where it routes and that it's now metered per request.

**How the code works - step by step**
- **`OpenAI(base_url=..., default_headers=...)`** - the one-line change: route through `oai.helicone.ai/v1` and attach the `Helicone-Auth` bearer header.
- **`client.chat.completions.create(...)`** - an ordinary capped chat call with a 20s timeout.
- **`r.usage`** - reads back `prompt_tokens` / `completion_tokens`, which are now also logged on the Helicone dashboard.
- **`try/except`** - prints a clean failure message instead of crashing if the call errors.

**In one line:** change the URL, keep the call, gain an itemized bill.

**What you'll see:** On success prints something like "in=14 out=22 (logged to Helicone)"; the token counts also appear on the Helicone dashboard tagged with cost and latency.

## 7 - Token optimization (two of the six levers)

Observability shows WHERE money goes; these techniques shrink each line item. None is dramatic alone - they compound, and they all attack the expensive side of the meter. This cell demonstrates two of the six: cap `max_tokens` so the model can't ramble, and compress the system prompt that's re-billed on every single request.

In [ ]:
# Step 6: two of the six techniques - cap max_tokens and compress the system prompt.
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")  # format comparison only, not a Claude bill
def ct(t):
    return len(enc.encode(t))

# Anti-pattern: a chatty, uncapped prompt re-billed on every single request.
bloated = "You are a very helpful, friendly and knowledgeable assistant. Please " \
          "always be polite and thorough in every single one of your answers."

# Fix: telegraphic system prompt + explicit output cap so the model can't ramble.
tight = "Role: support bot. Be terse. Answer only what is asked."
MAX_TOKENS = 150  # cap the expensive output lane

print(f"system prompt: {ct(bloated)} -> {ct(tight)} tokens")
print(f"saved {ct(bloated) - ct(tight)} tokens/call, output capped at {MAX_TOKENS}")
# Output:
#   system prompt: 30 -> 12 tokens
#   saved 18 tokens/call, output capped at 150

**Explanation**

A measurement cell again - tiktoken counts a bloated vs a telegraphic system prompt and sets an output cap. No model call; it's showing the before/after of two habits, not making a request.

**How the code works - step by step**
- **`ct()`** - the tiktoken helper (format comparison only, not a Claude bill).
- **`bloated`** - a chatty, uncapped system prompt re-billed every request.
- **`tight`** - the same instruction reduced to a terse, telegraphic form.
- **`MAX_TOKENS = 150`** - an explicit cap on the expensive output lane.
- **the prints** - report the system-prompt token drop and the per-call saving plus the output cap.

**In one line:** cap the output, compress the prompt - both pay off on every request.

**What you'll see:** Prints "system prompt: 30 -> 12 tokens" and "saved 18 tokens/call, output capped at 150".

## 8 - Prompt caching + batch (stack to 95% off)

Everything so far reduced what you send; the providers themselves offer the two biggest levers left, and they multiply. Prompt caching bills input the provider has already seen at ~10%, and the Batch API halves the rest for non-real-time work. This cell marks a large system block for caching with `cache_control` and notes the batch stacking math.

> **Needs an Anthropic API key** (set `ANTHROPIC_API_KEY`) and network access.

In [ ]:
# Step 7: prompt caching (cache_control on the big system block) + a batch note.
# Cached reads bill at about 10% on Anthropic; batch halves the rest -> they multiply.
import os
from anthropic import Anthropic
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

BIG_SYSTEM = "<5000-token policy + product manual reused on every request>"
try:
    r = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=200,
        system=[{"type": "text", "text": BIG_SYSTEM,
                 "cache_control": {"type": "ephemeral"}}],  # write about 1.25x, reads about 0.1x
        messages=[{"role": "user", "content": "What is the refund window?"}],
        timeout=30,
    )
    print(r.usage)  # 1st call WRITES the cache; cache_read_input_tokens fills in on the 2nd+ call
except Exception as e:
    print(f"call failed: {e}")
# Note: send non-realtime jobs via the Batch API (about 50% off). Cache 0.1x times batch 0.5x = 0.05x (about 95% off).
# Output: (first call - the system block is written to cache, not yet read)
#   Usage(input_tokens=12, cache_creation_input_tokens=5000, cache_read_input_tokens=0, output_tokens=18)

**Explanation**

An API-configuration demo showing the single most valuable Anthropic cost setting: tag a big reused system block as ephemeral cache so second-and-later calls read it cheaply. The commentary makes the multiplicative math explicit - cache 0.1x times batch 0.5x is 0.05x, about 95% off.

**How the code works - step by step**
- **`BIG_SYSTEM`** - a placeholder for the large (~5000-token) policy/manual reused on every request.
- **`system=[{... "cache_control": {"type": "ephemeral"}}]`** - marks that block for caching; writes cost ~1.25x, reads ~0.1x.
- **`messages.create(...)`** - a normal capped message with a 30s timeout.
- **`r.usage`** - the first call WRITES the cache (`cache_creation_input_tokens`), and `cache_read_input_tokens` fills in on later calls.
- **the batch note** - non-realtime jobs go via the Batch API (~50% off) and stack multiplicatively with caching.

**In one line:** cache the reused prefix, batch the non-urgent work, pay a twentieth.

**What you'll see:** On the first call prints a usage object with `cache_creation_input_tokens=5000` and `cache_read_input_tokens=0` - the system block is being written to cache, and reads (the discount) show up only on the second-and-later identical calls.

## 9 - Tiered routing (scaling the router)

Two-tier routing was the training wheels; production fleets run three or four tiers. This cell shows the routing skeleton with a deliberately cheap difficulty signal - a length-and-keyword heuristic - standing in for the classifier (RouteLLM) you'd use in production. The lesson's key metric to remember here is cost-per-successful-output, not cost-per-token.

In [ ]:
# Step 8: a tiered router - simple queries go cheap, complex ones go premium.
# The difficulty signal here is a cheap heuristic; production uses a classifier (RouteLLM).
CHEAP = "gemini-2.5-flash-lite"   # about $0.10/M - FAQ, classification
PREMIUM = "claude-sonnet-4-6"     # about $3.00/M - reasoning, synthesis

HARD_HINTS = ("design", "architect", "prove", "debug", "optimize")
def difficulty(query):
    if len(query.split()) > 30 or any(h in query.lower() for h in HARD_HINTS):
        return "complex"
    return "simple"

def route(query):
    return PREMIUM if difficulty(query) == "complex" else CHEAP

for q in ["What are your hours?", "Design a distributed system for millions of users"]:
    print(f"[{route(q):22s}] {q[:38]}")
# Output:
#   [gemini-2.5-flash-lite ] What are your hours?
#   [claude-sonnet-4-6     ] Design a distributed system for millio

**Explanation**

A pure-logic routing cell with no API call: a `difficulty` heuristic labels each query simple or complex, and `route` maps that to a cheap or premium model. It exists to show the decision boundary, not to make requests.

**How the code works - step by step**
- **`CHEAP / PREMIUM`** - the two model tiers with their approximate $/M input prices.
- **`HARD_HINTS`** - keywords (design, architect, prove, debug, optimize) that flag a complex query.
- **`difficulty()`** - returns "complex" if the query is over 30 words or contains a hint word, else "simple".
- **`route()`** - maps "complex" to PREMIUM and everything else to CHEAP.
- **the loop** - routes two example queries and prints the chosen model per query.

**In one line:** a cheap heuristic picks the tier; a real classifier replaces it in production.

**What you'll see:** Prints "What are your hours?" routed to gemini-2.5-flash-lite and "Design a distributed system..." routed to claude-sonnet-4-6.

## 10 - Budget enforcement (the prepaid meter)

Everything above optimizes the expected bill; this cell handles the unexpected one - the retry loop at 3 AM, the leaked key, the viral feature. Alerts only inform; enforcement protects. It implements graduated thresholds at the gateway: downgrade the model near the cap, then hard-block at 100% (because provider-native budgets are now soft-only).

In [ ]:
# Step 9: a budget guard - graduated steps, then a hard block at the daily cap.
# Alert, then downgrade the model near the cap, then block (the gateway, not the provider).
DAILY_CAP_USD = 50.0

class BudgetGuard:
    def __init__(self):
        self.spent = 0.0
    def check(self, requested_model, est_cost):
        if self.spent >= DAILY_CAP_USD:
            raise RuntimeError("budget exhausted - request blocked (HTTP 429)")
        frac = self.spent / DAILY_CAP_USD
        model = requested_model
        if frac >= 0.90:  # downgrade before the hard wall
            model = "gemini-2.5-flash-lite"
        self.spent += est_cost
        return model, frac

g = BudgetGuard()
g.spent = 46.0
print(g.check("claude-sonnet-4-6", 1.0))  # spent past 90 percent -> downgrade
try:
    g.spent = 50.0
    g.check("claude-sonnet-4-6", 1.0)
except RuntimeError as e:
    print(e)
# Output:
#   ('gemini-2.5-flash-lite', 0.92)
#   budget exhausted - request blocked (HTTP 429)

**Explanation**

A pure-logic guard class, no API call: it tracks spend against a daily cap and returns a possibly-downgraded model or raises when exhausted. This is the gateway-side enforcement the lesson insists you own, since OpenAI's native limits no longer hard-stop requests.

**How the code works - step by step**
- **`DAILY_CAP_USD = 50.0`** - the hard daily ceiling.
- **`__init__`** - starts the running `spent` total at zero.
- **`check()`** - raises a 429-style `RuntimeError` if already at the cap; otherwise computes the spent fraction, downgrades to flash-lite once past 90%, adds the estimated cost, and returns the (possibly downgraded) model plus the fraction.
- **the demo** - forces `spent` to 46.0 to trigger the downgrade, then to 50.0 to trigger the block.

**In one line:** dim the lights at 90%, cut the power at 100% - graduated, then hard.

**What you'll see:** Prints the downgrade tuple `('gemini-2.5-flash-lite', 0.92)` at 92% spend, then "budget exhausted - request blocked (HTTP 429)" once the cap is reached.

## 11 - India landed cost (GST + the Indic token tax)

The final layer is the one finance sees first: the gap between the sticker price on a pricing page and the landed price on an Indian invoice. This cell computes two of the biggest gaps - the Indic tokenization tax (Hindi packs far more tokens per word on English-trained tokenizers) and 18% GST - on a 100M-word/month workload.

In [ ]:
# Step 10: landed cost in India - Indic token tax + 18% GST on a monthly estimate.
# Hindi packs about 1.89 tokens/word on English-trained tokenizers vs about 1.16 for English.
EN_TOK_PER_WORD = 1.16
HI_TOK_PER_WORD = 1.89   # the Indic tokenization tax
GST = 0.18
PRICE_PER_M_USD = 2.50   # input price, $/M tokens
USD_TO_INR = 93.0

words_per_month = 100_000_000  # 100M Hindi words
en_tokens = words_per_month * EN_TOK_PER_WORD
hi_tokens = words_per_month * HI_TOK_PER_WORD
indic_tax_pct = (hi_tokens / en_tokens - 1) * 100

base_usd = hi_tokens / 1e6 * PRICE_PER_M_USD
landed_inr = base_usd * USD_TO_INR * (1 + GST)  # add 18% GST

print(f"Indic token tax: +{indic_tax_pct:.0f}%")
print(f"Landed cost (incl 18% GST): Rs {landed_inr:,.0f}/month")
# Output:
#   Indic token tax: +63%
#   Landed cost (incl 18% GST): Rs 51,852/month

**Explanation**

A pure arithmetic cell, no API call: it converts monthly Hindi word volume into tokens at Hindi vs English rates, prices it, then adds forex and GST to get a rupee landed cost. It quantifies the "showroom price vs on-road price" gap for a foreign LLM API.

**How the code works - step by step**
- **`EN_TOK_PER_WORD` / `HI_TOK_PER_WORD`** - tokens-per-word for English (~1.16) vs Hindi (~1.89), the tokenization penalty.
- **`GST`, `PRICE_PER_M_USD`, `USD_TO_INR`** - the 18% tax rate, the input $/M price, and the exchange rate.
- **token conversion** - multiplies 100M words by each rate to get English- and Hindi-equivalent token counts.
- **`indic_tax_pct`** - the percent extra tokens Hindi costs over English.
- **`base_usd` and `landed_inr`** - prices the Hindi tokens in USD, then converts to rupees and adds 18% GST.

**In one line:** sticker tokens x the Indic surcharge x forex x GST = the number on the invoice.

**What you'll see:** Prints "Indic token tax: +63%" and "Landed cost (incl 18% GST): Rs 51,852/month".

You now have the full cost stack in code: a metered ledger, an honest cheap-first router, a semantic cache, prompt compression, one-line observability, the two multiplicative provider discounts (caching x batch), a gateway budget guard, and India landed-cost math. The through-line is one honest metric - cost-per-successful-output, not cost-per-token - and one habit: never price what you haven't metered. Next, Lesson 7.4 builds structured outputs and Lesson 7.6 puts tracing and evaluation (Langfuse) on top of the dashboards you started here.